# 06 — Module 3: BRAS (Blockchain Rule Alignment Score)

Does each XAI explanation align with known blockchain fraud patterns?
Two quantities are combined:

1. **RAS@k ↑** — fraction of the top-k features that belong to the set of
   domain-relevant features for the instance.
2. **DVR@k ↓** — fraction of instances whose top-k features contain at
   least one domain-contradictory feature.

The composite **BRAS** is `alpha * RAS + (1 - alpha) * (1 - DVR)` with
`alpha = 0.5`. BRAS is evaluated **only on fraud instances** (fraud
explanations are where domain alignment matters).

Rule sets live in `src/xai_blockchain_framework/rules/`:

- `elliptic_rules` — groups R1-R4 over feature position ranges
  (structural, amount, temporal, neighborhood).
- `ethereum_rules` — groups R1-R5 over named features (token fraud,
  temporal anomaly, mixing, volume, contract interactions).

**Coverage (14 configs):** same 8 ML and 6 GNN configurations as
Modules 1-2, evaluated on 200 fraud instances per dataset.


In [ ]:
import os
from datetime import datetime

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch_geometric.data import Data
import warnings
warnings.filterwarnings('ignore')

from xai_blockchain_framework.config import PATHS, set_seed
from xai_blockchain_framework.models import TemporalGCN, GraphSAGEModel, get_device
from xai_blockchain_framework.metrics import (
    evaluate_bras,
    rule_alignment_score,
    domain_violation_rate,
    bras_score,
    top_k_indices,
)
from xai_blockchain_framework.rules import (
    elliptic_rules,
    ethereum_rules,
    ETH_TEMPORAL, ETH_VOLUME, ETH_ERC20, ETH_DIVERSITY,
    ETH_CONTRACT, ETH_AMOUNT, ETH_RATIOS,
)

set_seed(42)

DEVICE = get_device()
RESULTS_PATH = str(PATHS.results_dir)
FIGURES_PATH = str(PATHS.figures_dir)
MODELS_PATH = str(PATHS.models_dir)
PROCESSED_PATH = str(PATHS.data_processed)
for p in [RESULTS_PATH, FIGURES_PATH, PROCESSED_PATH]:
    os.makedirs(p, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')

TOP_K = 5
ALPHA = 0.5
N_EVAL_BRAS = 200

print("=" * 70)
print("MODULE 3  BRAS")
print("=" * 70)
print(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"TOP_K={TOP_K}  ALPHA={ALPHA}  N_EVAL={N_EVAL_BRAS}")
print(f"Device: {DEVICE}")


## 1. Load models, data, and pre-computed attributions

Fraud-only subsets are derived from `xai_eval_indices_{dataset}.npy`
(the balanced 200-instance indices saved by notebook 03a).


In [ ]:
print("\n" + "=" * 70)
print("1. Loading")
print("=" * 70)

# ML models
rf_ell = joblib.load(os.path.join(MODELS_PATH, 'elliptic_rf.joblib'))
lgb_ell = joblib.load(os.path.join(MODELS_PATH, 'elliptic_lgb.joblib'))
rf_eth = joblib.load(os.path.join(MODELS_PATH, 'ethereum_rf.joblib'))
lgb_eth = joblib.load(os.path.join(MODELS_PATH, 'ethereum_lgb.joblib'))

# Splits
ell = np.load(os.path.join(PROCESSED_PATH, 'elliptic_splits.npz'))
eth = np.load(os.path.join(PROCESSED_PATH, 'ethereum_splits.npz'))
X_test_ell, y_test_ell = ell['X_test'], ell['y_test']
X_test_eth, y_test_eth = eth['X_test'], eth['y_test']

# Eval indices
eval_idx_ell = np.load(os.path.join(RESULTS_PATH, 'xai_eval_indices_elliptic.npy'))
eval_idx_eth = np.load(os.path.join(RESULTS_PATH, 'xai_eval_indices_ethereum.npy'))

# Fraud-only subset (plus positions inside the full 200-instance attribution arrays)
def fraud_positions(eval_idx, y_test, n_eval):
    mask = y_test[eval_idx] == 1
    indices = eval_idx[mask][:n_eval]
    positions = np.where(mask)[0][:n_eval]
    return indices, positions

fraud_idx_ell, fraud_pos_ell = fraud_positions(eval_idx_ell, y_test_ell, N_EVAL_BRAS)
fraud_idx_eth, fraud_pos_eth = fraud_positions(eval_idx_eth, y_test_eth, N_EVAL_BRAS)
print(f"  Fraud BRAS subsets  Elliptic: {len(fraud_idx_ell)}  Ethereum: {len(fraud_idx_eth)}")

# Attributions
shap_rf_ell = np.load(os.path.join(RESULTS_PATH, 'shap_values_rf_elliptic.npy'))
shap_lgb_ell = np.load(os.path.join(RESULTS_PATH, 'shap_values_lgb_elliptic.npy'))
shap_rf_eth = np.load(os.path.join(RESULTS_PATH, 'shap_values_rf_ethereum.npy'))
shap_lgb_eth = np.load(os.path.join(RESULTS_PATH, 'shap_values_lgb_ethereum.npy'))
lime_rf_ell = np.load(os.path.join(RESULTS_PATH, 'lime_attrs_rf_elliptic.npy'))
lime_lgb_ell = np.load(os.path.join(RESULTS_PATH, 'lime_attrs_lgb_elliptic.npy'))
lime_rf_eth = np.load(os.path.join(RESULTS_PATH, 'lime_attrs_rf_ethereum.npy'))
lime_lgb_eth = np.load(os.path.join(RESULTS_PATH, 'lime_attrs_lgb_ethereum.npy'))

# GNN
n_feat_ell = X_test_ell.shape[1]
tgcn = TemporalGCN(in_c=n_feat_ell).to(DEVICE)
tgcn.load_state_dict(torch.load(os.path.join(MODELS_PATH, 'elliptic_temporal_gcn.pt'), map_location=DEVICE))
tgcn.eval()
sage = GraphSAGEModel(in_c=n_feat_ell).to(DEVICE)
sage.load_state_dict(torch.load(os.path.join(MODELS_PATH, 'elliptic_graphsage.pt'), map_location=DEVICE))
sage.eval()

edge_index = torch.load(os.path.join(PROCESSED_PATH, 'elliptic_edge_index.pt'))
graph_data = Data(
    x=torch.tensor(ell['X_all_n'], dtype=torch.float32),
    y=torch.tensor(ell['y_all'], dtype=torch.long),
    edge_index=edge_index,
    train_mask=torch.tensor(ell['train_mask']),
    val_mask=torch.tensor(ell['val_mask']),
    test_mask=torch.tensor(ell['test_mask']),
).to(DEVICE)
X_gnn_all = graph_data.x.cpu().numpy()

gnnexp_tgcn = np.load(os.path.join(RESULTS_PATH, 'gnn_attrs_gnnexp_tgcn.npy'))
gnnexp_sage = np.load(os.path.join(RESULTS_PATH, 'gnn_attrs_gnnexp_sage.npy'))
ig_tgcn = np.load(os.path.join(RESULTS_PATH, 'gnn_attrs_ig_tgcn.npy'))
ig_sage = np.load(os.path.join(RESULTS_PATH, 'gnn_attrs_ig_sage.npy'))
glime_tgcn = np.load(os.path.join(RESULTS_PATH, 'gnn_attrs_glime_tgcn.npy'))
glime_sage = np.load(os.path.join(RESULTS_PATH, 'gnn_attrs_glime_sage.npy'))

gnn_node_idx = np.load(os.path.join(RESULTS_PATH, 'gnn_eval_node_indices.npy'))
y_all = ell['y_all']
gnn_fraud_mask = y_all[gnn_node_idx] == 1
gnn_fraud_idx = gnn_node_idx[gnn_fraud_mask][:N_EVAL_BRAS]
gnn_fraud_pos = np.where(gnn_fraud_mask)[0][:N_EVAL_BRAS]
print(f"  GNN fraud nodes: {len(gnn_fraud_idx)}")


## 2. Evaluate BRAS on ML baselines


In [ ]:
print("\n" + "=" * 70)
print("2. ML baselines")
print("=" * 70)

all_results: list[dict] = []

ML_CONFIGS = [
    ('RF',  'Elliptic', X_test_ell, fraud_idx_ell, fraud_pos_ell, shap_rf_ell,  'SHAP', elliptic_rules),
    ('LGB', 'Elliptic', X_test_ell, fraud_idx_ell, fraud_pos_ell, shap_lgb_ell, 'SHAP', elliptic_rules),
    ('RF',  'Ethereum', X_test_eth, fraud_idx_eth, fraud_pos_eth, shap_rf_eth,  'SHAP', ethereum_rules),
    ('LGB', 'Ethereum', X_test_eth, fraud_idx_eth, fraud_pos_eth, shap_lgb_eth, 'SHAP', ethereum_rules),
    ('RF',  'Elliptic', X_test_ell, fraud_idx_ell, fraud_pos_ell, lime_rf_ell,  'LIME', elliptic_rules),
    ('LGB', 'Elliptic', X_test_ell, fraud_idx_ell, fraud_pos_ell, lime_lgb_ell, 'LIME', elliptic_rules),
    ('RF',  'Ethereum', X_test_eth, fraud_idx_eth, fraud_pos_eth, lime_rf_eth,  'LIME', ethereum_rules),
    ('LGB', 'Ethereum', X_test_eth, fraud_idx_eth, fraud_pos_eth, lime_lgb_eth, 'LIME', ethereum_rules),
]

for model_name, dataset, X, indices, positions, attrs, xai_name, rule_fn in ML_CONFIGS:
    label = f"{model_name}-{dataset}-{xai_name}"
    # Keep only positions that fall inside this attribution matrix (LIME is shorter than SHAP)
    valid = positions < len(attrs)
    positions_v = positions[valid]
    indices_v = indices[valid]
    if len(positions_v) == 0:
        print(f"\n--- {label}: no valid attribution rows, skip ---")
        continue
    attrs_v = attrs[positions_v]
    out = evaluate_bras(X, attrs_v, indices_v, rule_fn=rule_fn, k=TOP_K, alpha=ALPHA)
    print(f"\n--- {label} ({out['N_eval']} fraud) ---")
    print(f"  RAS@{TOP_K}={out['RAS']:.4f}  DVR@{TOP_K}={out['DVR']:.4f}  BRAS={out['BRAS']:.4f}")
    all_results.append({
        'Model': model_name, 'Dataset': dataset, 'XAI': xai_name, 'Type': 'ML',
        **out,
    })


## 3. Evaluate BRAS on GNN models (Elliptic)


In [ ]:
print("\n" + "=" * 70)
print("3. GNN models")
print("=" * 70)

GNN_CONFIGS = [
    ('T-GCN', 'GNNExp',    gnnexp_tgcn),
    ('T-GCN', 'IG',        ig_tgcn),
    ('T-GCN', 'GraphLIME', glime_tgcn),
    ('SAGE',  'GNNExp',    gnnexp_sage),
    ('SAGE',  'IG',        ig_sage),
    ('SAGE',  'GraphLIME', glime_sage),
]

for model_name, xai_name, attrs in GNN_CONFIGS:
    label = f"{model_name}-{xai_name}"
    valid = gnn_fraud_pos < len(attrs)
    positions_v = gnn_fraud_pos[valid]
    indices_v = gnn_fraud_idx[valid]
    if len(positions_v) == 0:
        print(f"\n--- {label}: no fraud nodes, skip ---")
        continue
    attrs_v = attrs[positions_v]
    out = evaluate_bras(X_gnn_all, attrs_v, indices_v, rule_fn=elliptic_rules, k=TOP_K, alpha=ALPHA)
    print(f"\n--- {label} ({out['N_eval']} fraud nodes) ---")
    print(f"  RAS@{TOP_K}={out['RAS']:.4f}  DVR@{TOP_K}={out['DVR']:.4f}  BRAS={out['BRAS']:.4f}")
    all_results.append({
        'Model': model_name, 'Dataset': 'Elliptic', 'XAI': xai_name, 'Type': 'GNN',
        **out,
    })


## 4. Summary table


In [ ]:
results_df = pd.DataFrame(all_results)
print(results_df.to_string(index=False))


## 5. Visualizations


In [ ]:
print("\n" + "=" * 70)
print("5. Visualizations")
print("=" * 70)

COLORS_ML = {'SHAP': '#2ecc71', 'LIME': '#e74c3c'}
COLORS_GNN = {'GNNExp': '#3498db', 'IG': '#9b59b6', 'GraphLIME': '#f39c12'}

def get_color(row):
    return COLORS_ML.get(row['XAI'], '#95a5a6') if row['Type'] == 'ML' \
        else COLORS_GNN.get(row['XAI'], '#95a5a6')

labels = [f"{r['Model']}\n{r['XAI']}\n{r['Dataset']}" for _, r in results_df.iterrows()]
colors = [get_color(r) for _, r in results_df.iterrows()]


def _bar(metric, ylabel, title, out_name, ylim=(0, 1.05)):
    fig, ax = plt.subplots(figsize=(14, 7))
    values = results_df[metric].values
    bars = ax.bar(range(len(values)), values, color=colors, alpha=0.85,
                  edgecolor='black', linewidth=0.5)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, fontsize=8, rotation=45, ha='right')
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim(*ylim)
    for bar, val in zip(bars, values):
        ax.annotate(f'{val:.3f}', xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                    xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=7)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_PATH, out_name), dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  wrote {out_name}")


_bar('RAS',  f'Rule Alignment Score @{TOP_K}', f'Module 3  RAS@{TOP_K}', 'module3_ras_bar.png')
_bar('DVR',  f'Domain Violation Rate @{TOP_K}', f'Module 3  DVR@{TOP_K}', 'module3_dvr_bar.png')
_bar('BRAS', f'BRAS (alpha={ALPHA})', 'Module 3  BRAS', 'module3_bras_bar.png')


## 6. Radar profiles (ML vs GNN)


In [ ]:
categories = ['RAS', '1 - DVR', 'BRAS']
N_cat = len(categories)
angles = [n / float(N_cat) * 2 * np.pi for n in range(N_cat)]
angles += angles[:1]

def _radar(subset, color_map, title, out_name):
    if subset.empty:
        print(f"  skipped {out_name}: empty")
        return
    fig, ax = plt.subplots(figsize=(10, 8), subplot_kw=dict(projection='polar'))
    for _, row in subset.iterrows():
        vals = [row['RAS'], 1 - row['DVR'], row['BRAS']]
        vals += vals[:1]
        color = color_map.get(row['XAI'], '#95a5a6')
        label = f"{row['Model']}-{row['XAI']}" + ("" if row['Type'] == 'GNN' else f"-{row['Dataset']}")
        ax.plot(angles, vals, 'o-', linewidth=2, label=label, color=color, alpha=0.7)
        ax.fill(angles, vals, alpha=0.1, color=color)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=11)
    ax.set_ylim(0, 1)
    ax.set_title(title, fontsize=14, fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.0), fontsize=9)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_PATH, out_name), dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  wrote {out_name}")

_radar(results_df[results_df['Type'] == 'ML'], COLORS_ML,
       'ML models  BRAS profile (higher = better)', 'module3_radar_ml.png')
_radar(results_df[results_df['Type'] == 'GNN'], COLORS_GNN,
       'GNN models  BRAS profile (higher = better)', 'module3_radar_gnn.png')


## 7. Rule-group heatmap (Ethereum)

For each Ethereum XAI configuration, compute the fraction of the top-k
features that fall in each rule group (temporal, volume, ERC20, etc.).


In [ ]:
rule_groups = {
    'Temporal': set(ETH_TEMPORAL),
    'Volume': set(ETH_VOLUME),
    'ERC20': set(ETH_ERC20),
    'Diversity': set(ETH_DIVERSITY),
    'Contract': set(ETH_CONTRACT),
    'Amount': set(ETH_AMOUNT),
    'Ratios': set(ETH_RATIOS),
}

ETH_CONFIGS_HEATMAP = [
    ('RF-SHAP',  shap_rf_eth),
    ('LGB-SHAP', shap_lgb_eth),
    ('RF-LIME',  lime_rf_eth),
    ('LGB-LIME', lime_lgb_eth),
]

heatmap_data = []
heatmap_labels = []
for config_label, attrs in ETH_CONFIGS_HEATMAP:
    valid = fraud_pos_eth < len(attrs)
    positions_v = fraud_pos_eth[valid]
    if len(positions_v) == 0:
        continue
    attrs_v = attrs[positions_v]
    row_scores = []
    for group_name, group_set in rule_groups.items():
        group_ras = []
        for a in attrs_v:
            top = top_k_indices(a, TOP_K)
            group_ras.append(len(top & group_set) / TOP_K)
        row_scores.append(float(np.mean(group_ras)))
    heatmap_data.append(row_scores)
    heatmap_labels.append(config_label)

if heatmap_data:
    arr = np.array(heatmap_data)
    fig, ax = plt.subplots(figsize=(12, 6))
    im = ax.imshow(arr, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)
    ax.set_xticks(range(len(rule_groups)))
    ax.set_xticklabels(list(rule_groups.keys()), fontsize=10, rotation=45, ha='right')
    ax.set_yticks(range(len(heatmap_labels)))
    ax.set_yticklabels(heatmap_labels, fontsize=10)
    for i in range(len(heatmap_labels)):
        for j in range(len(rule_groups)):
            text = f'{arr[i, j]:.2f}'
            color = 'white' if arr[i, j] > 0.5 else 'black'
            ax.text(j, i, text, ha='center', va='center', fontsize=9, color=color)
    ax.set_title(f'Module 3  Alignment per rule category (Ethereum, top-{TOP_K})',
                 fontsize=14, fontweight='bold')
    plt.colorbar(im, ax=ax, label=f'Mean proportion of top-{TOP_K} features')
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_PATH, 'module3_heatmap_rules.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print("  wrote module3_heatmap_rules.png")


## 8. Save results


In [ ]:
results_df.to_csv(os.path.join(RESULTS_PATH, 'module3_bras_results.csv'), index=False)
summary = results_df[['Type', 'Model', 'Dataset', 'XAI',
                      'RAS', 'DVR', 'BRAS', 'N_eval']].copy()
summary = summary.sort_values('BRAS', ascending=False)
summary.to_csv(os.path.join(RESULTS_PATH, 'module3_bras_summary.csv'), index=False)

print(f"""
Files saved:
  module3_bras_results.csv     (full long-format table)
  module3_bras_summary.csv     (sorted by BRAS)
  module3_ras_bar.png / module3_dvr_bar.png / module3_bras_bar.png
  module3_radar_ml.png / module3_radar_gnn.png
  module3_heatmap_rules.png

Interpretation:
  RAS@{TOP_K} (up)    : top-{TOP_K} features align with domain rules.
  DVR@{TOP_K} (down)  : fraction of explanations listing a contradictory feature.
  BRAS (up)            : {ALPHA}*RAS + {1 - ALPHA}*(1 - DVR).

14 configurations evaluated (8 ML + 6 GNN).
""")
